In [ ]:
import yfinance as yf
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# ── Stock universe ────────────────────────────────────────────────────────────
bank_stock_list = ["HDFCBANK.NS", "SBIN.NS", "ICICIBANK.NS", "BANKBARODA.NS",
                   "UNIONBANK.NS", "INDUSINDBK.NS", "AXISBANK.NS"]
med_stock_list  = ["CIPLA.NS", "LAURUSLABS.NS", "AUROPHARMA.NS",
                   "SUNPHARMA.NS", "ZYDUSLIFE.NS", "ABBOTINDIA.NS"]
it_stock_list   = ["TCS.NS", "HCLTECH.NS", "OFSS.NS", "WIPRO.NS",
                   "BSOFT.NS", "TECHM.NS", "MPHASIS.NS"]

# ── Data-quality constants ────────────────────────────────────────────────────
TRADING_DAYS_PER_YEAR = 252
MIN_HISTORY_YEARS     = 8
MIN_DATA_POINTS       = MIN_HISTORY_YEARS * TRADING_DAYS_PER_YEAR   # ≈ 2016 days

# ── Download 10 years of data; keep only stocks with >= 8 years ───────────────
print(f"Downloading 10-year data — keeping stocks with ≥ {MIN_HISTORY_YEARS} years "
      f"({MIN_DATA_POINTS}+ trading days)\n")

med_stock_data = []
for ticker in med_stock_list:
    stock  = yf.Ticker(ticker)
    df     = pd.DataFrame(stock.history(period="10y"))
    df     = df.drop(columns=["High", "Low", "Open", "Volume"], errors='ignore')
    n_days = len(df)
    years  = n_days / TRADING_DAYS_PER_YEAR
    status = "PASS" if n_days >= MIN_DATA_POINTS else "SKIP"
    print(f"  [{status}]  {ticker:<18}  {n_days:>4} trading days  ({years:.1f} yrs)")
    if n_days >= MIN_DATA_POINTS:
        med_stock_data.append([ticker, df, df['Close'].std()])

print(f"\n{len(med_stock_data)} / {len(med_stock_list)} pharma stocks passed "
      f"the {MIN_HISTORY_YEARS}-year history filter.")

if len(med_stock_data) < 2:
    raise ValueError("Fewer than 2 stocks passed the filter — check your data source.")

# ── Summary statistics + price history plots ─────────────────────────────────
for s in med_stock_data:
    ticker, df = s[0], s[1]
    print(f"\n{ticker}  |  mean = {df['Close'].mean():.2f}  |  std = {df['Close'].std():.2f}  "
          f"|  {len(df)} days")

    plt.figure(figsize=(10, 4))
    plt.plot(df['Close'].values, linewidth=1, label=ticker)
    plt.title(f"{ticker} — 10-Year Closing Price (NSE)")
    plt.xlabel("Trading Days")
    plt.ylabel("Price (₹)")
    plt.grid(True, alpha=0.4)
    plt.legend()
    plt.tight_layout()
    plt.show()


In [ ]:
# ── Build aligned closing-price lists from filtered stocks ───────────────────
# All stocks may have slightly different lengths due to corporate actions / IPO
# dates.  Take the LAST min_len days from each so the data is temporally aligned
# (most recent shared history).

lengths = [len(s[1]) for s in med_stock_data]
min_len = min(lengths)
num_stocks = len(med_stock_data)

print("Filtered stocks and data lengths:")
for s, l in zip(med_stock_data, lengths):
    print(f"  {s[0]:<18}  {l} days  (using last {min_len})")

print(f"\nnum_stocks  = {num_stocks}")
print(f"Aligned length = {min_len} trading days  "
      f"({min_len / TRADING_DAYS_PER_YEAR:.1f} years)")

closing_price_list = []
for s in med_stock_data:
    prices = s[1]['Close'].values[-min_len:].tolist()   # last min_len days
    closing_price_list.append(prices)


In [ ]:
# ── Build price_list and define train / test split ───────────────────────────
price_list = []
for v in range(len(closing_price_list[0])):
    price_list.append([closing_price_list[u][v] for u in range(num_stocks)])

# Train on 5 years, test on 3 years  (needs >= 8 years total)
TRAIN_YEARS = 5
TEST_YEARS  = 3
train_size  = TRAIN_YEARS * TRADING_DAYS_PER_YEAR   # 1260
test_size   = TEST_YEARS  * TRADING_DAYS_PER_YEAR   # 756

assert len(price_list) >= train_size + test_size, (
    f"Need {train_size + test_size} days but only have {len(price_list)}. "
    "Increase MIN_HISTORY_YEARS or change period."
)

print(f"Total price data  : {len(price_list)} trading days  "
      f"({len(price_list)/TRADING_DAYS_PER_YEAR:.1f} years)")
print(f"Training split    : days    0 – {train_size-1:<4}  ({TRAIN_YEARS} years)")
print(f"Test split        : days {train_size} – {train_size+test_size-1:<4}  ({TEST_YEARS} years)")
print(f"Stocks in model   : {num_stocks}  →  "
      + ", ".join(s[0] for s in med_stock_data))


In [ ]:
import numpy as np
import tensorflow as tf
import tensorflow_probability as tfp

# ── Hyperparameters ───────────────────────────────────────────────────────────
episode_length = 50       # trading days per MC episode
gamma          = 0.99     # discount factor  (Eq. 7 in poster)
learning_rate  = 0.001
initial_cash   = 100_000

train_price_list = price_list[:train_size]
test_price_list  = price_list[train_size : train_size + test_size]

# ── Policy Network ────────────────────────────────────────────────────────────
# Parameterised by (mean, log-variance) of a Gaussian; softmax over sampled
# raw logits gives  π(a|s,θ)  — aligns with poster Eq. 4.
class PolicyNetwork(tf.keras.Model):
    def __init__(self, n_stocks):
        super().__init__()
        self.dense1       = tf.keras.layers.Dense(64, activation='relu')
        self.dense2       = tf.keras.layers.Dense(64, activation='relu')
        self.mean_layer   = tf.keras.layers.Dense(n_stocks, activation='softmax')
        self.logvar_layer = tf.keras.layers.Dense(n_stocks)

    def call(self, state):
        x = self.dense2(self.dense1(state))
        return self.mean_layer(x), self.logvar_layer(x)

policy    = PolicyNetwork(num_stocks)
optimizer = tf.keras.optimizers.Adam(learning_rate)

# ── Environment helpers ───────────────────────────────────────────────────────
def give_state(cash, holdings, prices):
    # State: [cash, 0,0,0,0,0, n1..n_k, c1..c_k]  (poster §REINFORCE Methodology)
    return np.concatenate([[cash, 0, 0, 0, 0, 0], holdings, prices]).astype(np.float32)

def sample_action(state, cash, holdings):
    prices       = state[-num_stocks:]
    st           = tf.convert_to_tensor(state[None, :], dtype=tf.float32)
    mean, logvar = policy(st)
    std          = tf.exp(0.5 * logvar)
    raw          = mean + tf.random.normal(tf.shape(mean)) * std
    probs_vec    = tf.nn.softmax(raw).numpy().flatten()   # poster Eq. 4

    action = []
    for i in range(num_stocks):
        max_sell = min(int(holdings[i]), 5)
        max_buy  = min(int(cash // prices[i]) if prices[i] > 1e-8 else 0, 5)
        opts     = np.arange(-max_sell, max_buy + 1)
        if len(opts) == 0:  action.append(0);            continue
        if len(opts) == 1:  action.append(int(opts[0])); continue
        p = np.clip(np.abs(probs_vec[i]) * np.ones(len(opts)), 1e-8, None)
        p = np.nan_to_num(p, nan=1.0 / len(opts))
        p /= p.sum()
        action.append(int(np.random.choice(opts, p=p)))

    action     = np.array(action)
    total_cost = sum(action[i] * prices[i] for i in range(num_stocks))
    if total_cost > cash + 1e-8:
        sc     = cash / (total_cost + 1e-8)
        action = np.array([int(np.floor(a * sc)) if a > 0 else a for a in action])
    return action

def step_env(cash, holdings, action, prices):
    new_cash = cash - sum(action[i] * prices[i] for i in range(num_stocks))
    new_h    = [holdings[i] + action[i] for i in range(num_stocks)]
    return new_cash, new_h

def port_val(cash, holdings, prices):
    return cash + sum(holdings[i] * prices[i] for i in range(num_stocks))

def log_ret(v_old, v_new):
    return float(np.log(v_new / v_old)) if v_old > 1e-8 else 0.0

def compute_returns(rewards, gamma):
    """Monte Carlo returns  Gt = Σ_{k=t+1}^{T} γ^{k-t-1} Rk  (poster Eq. 7)"""
    G  = np.zeros(len(rewards), dtype=np.float32)
    Gt = 0.0
    for t in reversed(range(len(rewards))):
        Gt = rewards[t] + gamma * Gt
        G[t] = Gt
    return G

# ── Evaluation helper ─────────────────────────────────────────────────────────
def evaluate_policy(data, label, base_day=0):
    """
    Run the trained policy (frozen — no gradient updates) through `data`.
    Records portfolio value at the end of each trading year and prints a
    formatted annual-return table.
    Returns list of portfolio-value snapshots (index 0 = initial capital).
    """
    cash, holdings, v0 = initial_cash, [0] * num_stocks, initial_cash
    snapshots = [initial_cash]

    for day_idx, prices in enumerate(data):
        state  = give_state(cash, holdings, prices)
        action = sample_action(state, cash, holdings)
        cash, holdings = step_env(cash, holdings, action, prices)
        v = port_val(cash, holdings, prices)
        if (day_idx + 1) % TRADING_DAYS_PER_YEAR == 0:
            snapshots.append(v)

    n_years = len(snapshots) - 1
    W = 64
    print(f"\n{'═'*W}")
    print(f"  {label}")
    print(f"{'─'*W}")
    print(f"  {'Year':<6}  {'Day range':<13}  {'Start (₹)':>13}  {'End (₹)':>13}  {'Return':>8}")
    print(f"  {'────':<6}  {'─────────':<13}  {'──────────':>13}  {'──────────':>13}  {'──────':>8}")

    cumulative = 1.0
    for yr in range(n_years):
        sv   = snapshots[yr]
        ev   = snapshots[yr + 1]
        pct  = (ev - sv) / sv * 100
        cumulative *= (1 + pct / 100)
        d0   = base_day + yr * TRADING_DAYS_PER_YEAR
        d1   = d0 + TRADING_DAYS_PER_YEAR - 1
        mark = "▲" if pct >= 0 else "▼"
        print(f"  {mark} Yr {yr+1:<2}  {d0:>4} – {d1:<4}    "
              f"{sv:>13,.0f}  {ev:>13,.0f}  {pct:>+7.2f}%")

    total_pct = (cumulative - 1) * 100
    print(f"{'─'*W}")
    print(f"  Cumulative return  :  {total_pct:>+8.2f}%")
    print(f"  Final portfolio    :  ₹ {snapshots[-1]:>12,.2f}")
    print(f"{'═'*W}")
    return snapshots

# ═════════════════════════════════════════════════════════════════════════════
#  PHASE 1 — REINFORCE TRAINING  (Monte Carlo Policy Gradient)
#  Data : first 5 years  ({train_size} trading days)
# ═════════════════════════════════════════════════════════════════════════════
W = 64
print("═"*W)
print("  PHASE 1 — REINFORCE TRAINING  (Monte Carlo Policy Gradient)")
print(f"  Data: days 0–{train_size-1}  ({TRAIN_YEARS} years,  {len(train_price_list)} days)")
print("═"*W)

num_train_eps = len(train_price_list) // episode_length

for ep in range(num_train_eps):
    s = ep * episode_length
    e = s + episode_length
    c, h, v0 = initial_cash, [0] * num_stocks, initial_cash
    S_buf, A_buf, R_buf = [], [], []

    # ── Step 1: collect full episode trajectory (St, At, Rt) ─────────────────
    for t in range(s, e):
        prices = train_price_list[t]
        state  = give_state(c, h, prices)
        action = sample_action(state, c, h)
        c, h   = step_env(c, h, action, prices)
        new_v  = port_val(c, h, prices)
        R_buf.append(log_ret(v0, new_v));  v0 = new_v
        S_buf.append(state);  A_buf.append(action)

    # ── Step 2: Monte Carlo returns  Gt  (poster Eq. 7) ──────────────────────
    G = compute_returns(R_buf, gamma)
    G = (G - G.mean()) / (G.std() + 1e-8)   # normalise for stability

    # ── Step 3: REINFORCE update  θ ← θ + α·Gt·∇θ log π(At|St,θ)  (Eq. 8) ──
    with tf.GradientTape() as tape:
        total_loss = 0.0
        for t in range(episode_length):
            st_t         = tf.convert_to_tensor(S_buf[t][None, :], dtype=tf.float32)
            mean, logvar = policy(st_t)
            std          = tf.exp(0.5 * logvar)
            dist         = tfp.distributions.Normal(mean, std)
            log_pi       = tf.reduce_sum(dist.log_prob(
                               tf.convert_to_tensor(A_buf[t], dtype=tf.float32)))
            total_loss  += -log_pi * float(G[t])

    grads = tape.gradient(total_loss, policy.trainable_variables)
    optimizer.apply_gradients(zip(grads, policy.trainable_variables))

    if (ep + 1) % 5 == 0 or ep == 0:
        print(f"  Ep {ep+1:>3}/{num_train_eps}  |  days {s:>4}–{e-1:<4}  |  "
              f"Σ reward = {sum(R_buf):+.4f}  |  ep-end portfolio ₹{v0:>10,.2f}")

print(f"\n  Training complete — {num_train_eps} episodes over {len(train_price_list)} days.")

# ═════════════════════════════════════════════════════════════════════════════
#  PHASE 2 — EVALUATION  (policy frozen — no gradient updates)
#  In-sample  : training data (5 years)
#  Out-of-sample : test data   (3 years)
# ═════════════════════════════════════════════════════════════════════════════
print("\n\n" + "═"*W)
print("  PHASE 2 — EVALUATION  (policy frozen — no gradient updates)")
print("═"*W)

train_snap = evaluate_policy(
    train_price_list,
    f"IN-SAMPLE ANNUAL RETURNS  (Training data — {TRAIN_YEARS} years)",
    base_day=0
)

test_snap = evaluate_policy(
    test_price_list,
    f"OUT-OF-SAMPLE ANNUAL RETURNS  (Test data — {TEST_YEARS} years)",
    base_day=train_size
)

# ── Summary table ─────────────────────────────────────────────────────────────
print(f"\n{'═'*W}")
print(f"  SUMMARY")
print(f"{'─'*W}")
train_total = (train_snap[-1] / initial_cash - 1) * 100
test_total  = (test_snap[-1]  / initial_cash - 1) * 100
print(f"  Initial capital            :  ₹ {initial_cash:>12,.2f}")
print(f"  In-sample  final value     :  ₹ {train_snap[-1]:>12,.2f}  "
      f"({train_total:>+8.2f}%  over {TRAIN_YEARS} yrs)")
print(f"  Out-of-sample final value  :  ₹ {test_snap[-1]:>12,.2f}  "
      f"({test_total:>+8.2f}%  over {TEST_YEARS} yrs)")
print(f"{'═'*W}")
